# Calibration 04-2 — L-BFGS-B Fit (2019-2020, 7 age groups)

**설정** (NM 과 동일):
- 22D 파라미터 + Gaussian seasonality + first_peak_only + auto-seed + immunity=0.3
- L-BFGS-B: gradient-based (finite difference). 빠르지만 local minimum 가능.
- `max_iterations=2000`

**예상 시간**: 30-90분.

**주의**: 좁힌 bounds + min_rate=0.1 적용 후 L-BFGS-B 가 corner solution 으로 도망가는 문제 해결됐는지 확인할 시즌. 이전 fit (cosine_archive) 의 β=0.001 corner 와 비교.

출력: `outputs/calibration/2019-2020_by_age_LBFGS_<timestamp>.json` + 시각화 PNG (timestamped, 기존 파일 보존).

In [1]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

from kt_data import ILI_AGE_GROUPS
from kt_epimodel.calibration.ili_target import (
    load_ili_target_by_age, simulation_to_ili_by_age,
)
from kt_epimodel.calibration.optimizer import (
    optimize_calibration_by_age, save_result, load_result,
)
from kt_epimodel.calibration.simple_model import (
    build_aggregated_inputs, simulate_aggregated,
    estimate_initial_infected_from_ili,
)
from kt_epimodel.model.parameters import (
    DiseaseParameters, ModelParameters,
)

OUT = Path('../outputs/calibration'); OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

## 1. ILI target 확인

In [2]:
target_age = load_ili_target_by_age('2019-2020', first_peak_only=True, first_peak_end_week=26)
weeks = target_age['week_in_season']

## 2. L-BFGS-B fit (max_iter=2000)

Gradient-based — finite difference로 ∂NLL/∂param 추정 후 quasi-Newton step.
Bounds 적용 (scipy 1.7+). 빠르나 local minimum 도달 가능.

**Dead zone 회피** — vax flux 버그 수정 후 옛 β=0.05 default 는 R₀<1 영역이라 시뮬 결과가 거의 0 으로 클립되어 gradient 가 정확히 0 으로 잡힘 → L-BFGS-B 가 24 evals 만에 즉시 종료. `CalibrationParameters` default β 를 0.3 으로 올려 해결 (epidemic 영역).

## L-BFGS-B fit (seed 100배 축소 — corner 탈출 시도)

이전 fit 진단 결과 seed 가 ILI prediction 의 99.9% 를 차지 (amplification=1.0).
즉, optimizer 가 "no transmission + seed decay" 만으로 fit 하는 corner solution.
**Seed 를 10× 추가 축소 (총 100× 대비 원본)** 하여 epidemic 발생을 강제.

| 인자 | 변경 전 | 현재 |
|---|---|---|
| `gamma_report_assumed` | 20.0 | **200.0** |
| → I_0 (전 연령) | ~3,837 | ~384 |
| → E_0 (전 연령) | ~1,918 | ~192 |
| → 합계 (E+I) | ~5,755 | ~576 |

`seed_e_factor=0.5` 유지 (E/I 정상상태 비율).

**해석**

- Seed 가 너무 작아 그 자체로는 ILI baseline 못 만듦
- Optimizer 가 transmission (β, φ, seasonality) 으로 fit 하도록 강제됨
- R0 > 1 영역으로 끌어내는 압력


In [ ]:
from datetime import datetime
import bisect
from scipy.optimize import minimize

from kt_epimodel.calibration.loss import make_loss_function_by_age
from kt_epimodel.calibration.ili_target import load_ili_target_by_age
from kt_epimodel.calibration.param_vector import (
    initial_guess, get_bounds_vector, get_param_names, vector_to_params,
)
from kt_epimodel.calibration.optimizer import CalibrationResult
from kt_epimodel.calibration.simple_model import (
    build_aggregated_inputs, estimate_initial_infected_from_ili,
)

# ---- mirror optimize_calibration_by_age setup so we can wrap loss ----
SEASON = '2019-2020'
GAMMA_REPORT_ASSUMED = 200.0
SEED_E_FACTOR = 0.5
INITIAL_IMMUNITY = 0.3
T_SPAN = (0.0, 364.0)
FIRST_PEAK_ONLY = True
FIRST_PEAK_END_WEEK = 26
MAX_ITER = 2000

TOP_K = 20   # keep top-K lowest-NLL evals (potential local minima candidates)

base_params = ModelParameters()
target_by_age = load_ili_target_by_age(
    SEASON, first_peak_only=FIRST_PEAK_ONLY, first_peak_end_week=FIRST_PEAK_END_WEEK,
)
inputs_full = build_aggregated_inputs()
seed_by_age_arr = estimate_initial_infected_from_ili(
    SEASON, inputs_full['pop_15'].flatten(),
    gamma_report_assumed=GAMMA_REPORT_ASSUMED,
)
seed_total_effective = float(seed_by_age_arr.sum())
print(f"Estimated seed total: {seed_total_effective:,.0f}")

loss_inner = make_loss_function_by_age(
    target_by_age, inputs_full, base_params,
    seed_total=seed_total_effective, seed_by_age=seed_by_age_arr,
    seed_e_factor=SEED_E_FACTOR,
    initial_immunity=INITIAL_IMMUNITY,
    t_span=T_SPAN, verbose=False,
)

# Keep top-K lowest-NLL evaluations ever seen.
# top_k is sorted ascending by nll. Each entry: (nll, eval_idx, vec).
top_k: list[tuple[float, int, np.ndarray]] = []
top_k_keys: list[float] = []
eval_count = [0]
def loss_topk_capture(vec):
    eval_count[0] += 1
    v = float(loss_inner(np.asarray(vec, dtype=np.float64)))
    if len(top_k) < TOP_K or v < top_k_keys[-1]:
        pos = bisect.bisect_left(top_k_keys, v)
        top_k_keys.insert(pos, v)
        top_k.insert(pos, (v, eval_count[0], np.asarray(vec, dtype=np.float64).copy()))
        if len(top_k) > TOP_K:
            top_k.pop()
            top_k_keys.pop()
        print(f"  [eval {eval_count[0]:>5}]  NLL = {v:+.4f}  → rank {pos+1}/{TOP_K}")
    return v

initial_vec = initial_guess()
nll_initial = loss_topk_capture(initial_vec)
print(f"=== Optimizing {SEASON} (by_age, 7 groups) with L-BFGS-B ===")
print(f"Initial NLL: {nll_initial:.2f}")

t0 = time.time()
sol = minimize(
    loss_topk_capture, initial_vec,
    method='L-BFGS-B', bounds=get_bounds_vector(),
    options={'maxiter': MAX_ITER},
)
elapsed = time.time() - t0
print(f"\nTotal elapsed: {elapsed/60:.1f} min  ({eval_count[0]} evals, "
      f"top-{len(top_k)} recorded)")
print(f"scipy message: {sol.message}")
print(f"final NLL: {sol.fun:.2f}  |  best NLL: {top_k[0][0]:.2f}")

# Save each rank as a separate JSON  (canonical rank-1 also saved without the rank suffix)
RUN_TAG = datetime.now().strftime('%Y%m%d_%H%M%S')

def _make_result(nll_v: float, eval_idx: int, vec: np.ndarray) -> CalibrationResult:
    cal, amp_, base_, sigma_, peak_ = vector_to_params(vec)
    return CalibrationResult(
        season=f'{SEASON}_by_age', method='L-BFGS-B',
        success=bool(sol.success), nll=float(nll_v), nll_initial=float(nll_initial),
        calibration=cal,
        seasonality_amp=amp_, seasonality_base=base_,
        seasonality_sigma=sigma_, seasonality_peak_day=peak_,
        seasonality_mode=base_params.disease.seasonality_mode,
        vector=vec, n_evaluations=int(eval_idx),   # eval at which this rank was hit
        elapsed_seconds=float(elapsed), message=str(sol.message),
        seed_total=seed_total_effective, initial_immunity=INITIAL_IMMUNITY,
        initial_vaccinated_fraction=0.0,
        first_peak_only=FIRST_PEAK_ONLY, first_peak_end_week=FIRST_PEAK_END_WEEK,
        use_data_seed=True, seed_by_age=seed_by_age_arr.tolist(),
        gamma_report_assumed=GAMMA_REPORT_ASSUMED,
    )

result_lbfgs = _make_result(*top_k[0])
LBFGS_JSON = OUT / f'2019-2020_by_age_LBFGS_{RUN_TAG}.json'
save_result(result_lbfgs, LBFGS_JSON)

RANK_DIR = OUT / f'2019-2020_by_age_LBFGS_{RUN_TAG}_topk'
RANK_DIR.mkdir(exist_ok=True)
saved_rank_paths = []
for r, (nll_v, ev_i, vec) in enumerate(top_k, start=1):
    p = RANK_DIR / f'rank{r:02d}.json'
    save_result(_make_result(nll_v, ev_i, vec), p)
    saved_rank_paths.append(p)

TOPK_NPZ = OUT / f'2019-2020_by_age_LBFGS_{RUN_TAG}_top{TOP_K}.npz'
np.savez(TOPK_NPZ,
         rank_nll=np.asarray([t[0] for t in top_k]),
         rank_eval=np.asarray([t[1] for t in top_k]),
         rank_vec=np.asarray([t[2] for t in top_k]),
         param_names=np.asarray(get_param_names()))

print(f"Saved (rank-1 canonical):  {LBFGS_JSON.name}")
print(f"Saved per-rank JSONs:      {RANK_DIR.name}/rank01.json .. rank{len(top_k):02d}.json")
print(f"Saved top-K npz summary:   {TOPK_NPZ.name}")

In [ ]:
# ---- Top-K analysis: rank table + parameter spread (multiple local minima?) ----
rank_nll = np.asarray([t[0] for t in top_k])
rank_eval = np.asarray([t[1] for t in top_k])
rank_vec = np.asarray([t[2] for t in top_k])
names = get_param_names()
K = len(top_k)

print(f"Top-{K} lowest-NLL evals "
      f"(NLL range {rank_nll[0]:.2f} .. {rank_nll[-1]:.2f}, gap {rank_nll[-1]-rank_nll[0]:.2f}).\n")

# Table: rank, eval, NLL, key params
print(f"{'rank':>4}  {'eval':>6}  {'NLL':>10}  "
      f"{'β_h':>6}  {'β_w':>6}  {'β_s':>6}  {'β_o':>6}  "
      f"{'γ_r':>6}  {'amp':>6}  {'base':>6}  {'σ':>6}  {'pk':>5}")
for r in range(K):
    v = rank_vec[r]
    print(f"{r+1:>4}  {rank_eval[r]:>6}  {rank_nll[r]:>10.2f}  "
          f"{v[0]:>6.3f}  {v[1]:>6.3f}  {v[2]:>6.3f}  {v[3]:>6.3f}  "
          f"{v[18]:>6.3f}  {v[19]:>6.3f}  {v[20]:>6.3f}  {v[21]:>6.1f}  {v[22]:>5.0f}")

# Cluster top-K by parameter distance (normalized to bounds width) — different basins?
bnds = np.asarray(get_bounds_vector(), dtype=np.float64)   # (23, 2)
width = bnds[:, 1] - bnds[:, 0]
vec_norm = (rank_vec - bnds[:, 0]) / width   # each param in [0, 1]
# Pairwise L∞ distance (max coordinate diff after normalization)
n = len(rank_vec)
dist = np.zeros((n, n))
for i in range(n):
    dist[i] = np.max(np.abs(vec_norm - vec_norm[i]), axis=1)

# Greedy clustering: walk in rank order, assign to nearest seen cluster within threshold.
CLUSTER_THRESH = 0.05    # 각 차원 5% 이내면 같은 basin 으로 묶음
cluster_id = np.full(n, -1, dtype=int)
next_cid = 0
for i in range(n):
    for j in range(i):
        if dist[i, j] < CLUSTER_THRESH:
            cluster_id[i] = cluster_id[j]
            break
    if cluster_id[i] == -1:
        cluster_id[i] = next_cid
        next_cid += 1
print(f"\nDetected {next_cid} cluster(s) at param-distance < {CLUSTER_THRESH} "
      f"(normalized to bounds width):")
for c in range(next_cid):
    idxs = np.where(cluster_id == c)[0]
    print(f"  cluster {c}: ranks {[i+1 for i in idxs]}  "
          f"NLL {rank_nll[idxs].min():.2f} .. {rank_nll[idxs].max():.2f}")

# Plot
fig, axes = plt.subplots(2, 2, figsize=(14, 7))
colors = plt.cm.tab10(cluster_id % 10)

ax = axes[0, 0]
ax.scatter(np.arange(1, K + 1), rank_nll, c=colors, s=40, edgecolors='k', lw=0.5)
ax.set_xlabel('rank'); ax.set_ylabel('NLL'); ax.set_title(f'Top-{K} NLL (color = cluster)')
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
ax.scatter(rank_eval, rank_nll, c=colors, s=40, edgecolors='k', lw=0.5)
ax.set_xlabel('eval'); ax.set_ylabel('NLL'); ax.set_title('When each rank was found')
ax.grid(True, alpha=0.3)

# β scatter — βh vs βw, colored by cluster
ax = axes[1, 0]
ax.scatter(rank_vec[:, 0], rank_vec[:, 1], c=colors, s=60, edgecolors='k', lw=0.5)
for r in range(K):
    ax.annotate(str(r + 1), (rank_vec[r, 0], rank_vec[r, 1]), fontsize=8)
ax.set_xlabel('β_h'); ax.set_ylabel('β_w'); ax.set_title('Top-K in β_h × β_w')
ax.grid(True, alpha=0.3)

# seasonality scatter — amp vs σ
ax = axes[1, 1]
ax.scatter(rank_vec[:, 19], rank_vec[:, 21], c=colors, s=60, edgecolors='k', lw=0.5)
for r in range(K):
    ax.annotate(str(r + 1), (rank_vec[r, 19], rank_vec[r, 21]), fontsize=8)
ax.set_xlabel('amp'); ax.set_ylabel('σ'); ax.set_title('Top-K in amp × σ')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUT / f'2019-2020_by_age_LBFGS_top{TOP_K}_clusters_{RUN_TAG}.png', dpi=110)
plt.show()

## 3. Fit 결과 — 7 그룹 관측 vs 예측

In [ ]:
# Skip-fit mode: cell 6 (optimization) 을 건너뛰어도 시각화 가능.
# result_lbfgs / target_age / weeks 가 메모리에 없으면 디스크에서 로드.
def _latest_lbfgs_json() -> Path:
    """Return the newest 2019-2020_by_age_LBFGS*.json (timestamped or legacy)."""
    candidates = sorted(OUT.glob('2019-2020_by_age_LBFGS*.json'), key=lambda p: p.stat().st_mtime)
    if not candidates:
        raise FileNotFoundError("No L-BFGS-B result JSON found in outputs/calibration/")
    return candidates[-1]

if 'result_lbfgs' not in dir():
    latest = _latest_lbfgs_json()
    print(f"Loading: {latest.name}")
    result_lbfgs = load_result(latest)
if 'target_age' not in dir():
    target_age = load_ili_target_by_age('2019-2020', first_peak_only=True, first_peak_end_week=26)
if 'weeks' not in dir():
    weeks = target_age['week_in_season']

inputs = build_aggregated_inputs()
pop_15 = inputs['pop_15'].flatten()

disease = DiseaseParameters(
    seasonality_mode=result_lbfgs.seasonality_mode,
    seasonality_amp=result_lbfgs.seasonality_amp,
    seasonality_base=result_lbfgs.seasonality_base,
    seasonality_sigma=result_lbfgs.seasonality_sigma,
    seasonality_peak_day=result_lbfgs.seasonality_peak_day,
)
params = ModelParameters(disease=disease).with_calibration(result_lbfgs.calibration)

if result_lbfgs.use_data_seed:
    seed_by_age = estimate_initial_infected_from_ili(
        '2019-2020', pop_15,
        gamma_report_assumed=result_lbfgs.gamma_report_assumed,
    )
else:
    seed_by_age = None

sim = simulate_aggregated(
    params, inputs,
    seed_total=result_lbfgs.seed_total if not result_lbfgs.use_data_seed else 0.0,
    seed_by_age=seed_by_age,
    seed_e_factor=0.5,  # match fit cell (CalibrationResult does not persist this)
    initial_immunity=result_lbfgs.initial_immunity,
    t_span=(0.0, 364.0),
)
daily_inc = sim.daily_new_infection_by_age()  # Δ(E+I+R), excludes vax flux
predictions = simulation_to_ili_by_age(
    daily_inc, pop_15, result_lbfgs.calibration.gamma_report, n_weeks=52,
)

# Use a run tag for output PNG filenames (preserve prior figures)
PNG_TAG = RUN_TAG if 'RUN_TAG' in dir() else _latest_lbfgs_json().stem.replace('2019-2020_by_age_LBFGS_', '')

fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharex=True)
for i, ag in enumerate(ILI_AGE_GROUPS):
    ax = axes[i // 4, i % 4]
    ax.plot(weeks, target_age['ili_rates'][ag], 'ko-', markersize=3, label='observed')
    ax.plot(weeks, predictions[ag], 'b-', linewidth=2, label='predicted (L-BFGS-B)')
    ax.axvspan(26, 52, alpha=0.1, color='gray')
    ax.set_title(f'Age {ag}')
    ax.grid(True, alpha=0.3)
    if i % 4 == 0:
        ax.set_ylabel('ILI / 1000')
    if i // 4 == 1:
        ax.set_xlabel('Week')
axes[1, 3].axis('off')
axes[0, 0].legend(loc='upper right', fontsize=8)
fig.suptitle(
    f'2019-2020 L-BFGS-B fit (mode={result_lbfgs.seasonality_mode}) — '
    f'NLL {result_lbfgs.nll_initial:.0f} → {result_lbfgs.nll:.0f}',
    fontsize=13,
)
plt.tight_layout()
plt.savefig(OUT / f'2019-2020_by_age_LBFGS_fit_{PNG_TAG}.png', dpi=120)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
AGES = ['0-4','5-9','10-14','15-19','20-24','25-29','30-34','35-39',
        '40-44','45-49','50-54','55-59','60-64','65-69','70+']
axes[0].bar(np.arange(15), result_lbfgs.calibration.phi, color='C0')
axes[0].axhline(1.0, color='red', linestyle='--', label='reference')
axes[0].set_xticks(np.arange(15)); axes[0].set_xticklabels(AGES, rotation=45)
axes[0].set_ylabel(r'$\phi_a$')
axes[0].set_title('Age-specific susceptibility (L-BFGS-B fit)')
axes[0].legend(); axes[0].grid(True, alpha=0.3, axis='y')

channels = ['home', 'work', 'school', 'other']
betas = [result_lbfgs.calibration.beta_h, result_lbfgs.calibration.beta_w,
         result_lbfgs.calibration.beta_s, result_lbfgs.calibration.beta_o]
axes[1].bar(channels, betas, color=['C0', 'C1', 'C2', 'C3'])
axes[1].set_ylabel(r'$\beta_{ch}$')
axes[1].set_title(
    f'Channel β (γ_r={result_lbfgs.calibration.gamma_report:.3f}, '
    f'amp={result_lbfgs.seasonality_amp:.2f}, base={result_lbfgs.seasonality_base:.2f}, '
    f'σ={result_lbfgs.seasonality_sigma:.0f})'
)
axes[1].grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(OUT / f'2019-2020_by_age_LBFGS_phi_beta_{PNG_TAG}.png', dpi=120)
plt.show()

print(pl.DataFrame([{
    'method': result_lbfgs.method,
    'nll_initial': result_lbfgs.nll_initial,
    'nll_final': result_lbfgs.nll,
    'n_evals': result_lbfgs.n_evaluations,
    'elapsed_min': result_lbfgs.elapsed_seconds / 60,
    'beta_h': result_lbfgs.calibration.beta_h,
    'beta_w': result_lbfgs.calibration.beta_w,
    'beta_s': result_lbfgs.calibration.beta_s,
    'beta_o': result_lbfgs.calibration.beta_o,
    'gamma_report': result_lbfgs.calibration.gamma_report,
    'amp': result_lbfgs.seasonality_amp,
    'base': result_lbfgs.seasonality_base,
    'sigma': result_lbfgs.seasonality_sigma,
    'phi_min': float(result_lbfgs.calibration.phi.min()),
    'phi_max': float(result_lbfgs.calibration.phi.max()),
}]))